# Spoken Grammar Scoring Engine
## Automated Evaluation of Spoken English via Multimodal Stacking

---

### 📋 Brief Solution Report

#### 1. Objective & Challenge
The objective is to predict a continuous Grammar Score ranging from 0.0 to 5.0 (MOS Likert scale) from 45 to 60-second spoken audio clips (`.wav`).

#### 2. Core Methodology & Pipeline Architecture
Automated evaluation of spoken grammar presents a unique challenge: standard Automatic Speech Recognition (ASR) systems often "auto-correct" spoken grammar errors and strip out acoustic disfluencies (pauses, hesitations, articulation rate). To address this, we developed a **3-Stage Multimodal Stacking Pipeline**:

1. **Error-Preserving ASR (`faster-whisper-large-v3`):** 
   - Audio is transcribed with `condition_on_previous_text=False` to prevent language-model smoothing and retain verbatim grammatical mistakes, broken sentences, and repetitions.
2. **Syntactic Text Modeling (`DeBERTa-v3-base`):**
   - We fine-tune DeBERTa-v3 using 5-fold cross-validation.
   - We implement **differential learning rates** (`1e-4` for the randomly initialized regression head, `1e-5` for the pretrained encoder) and full FP32 precision to ensure numerical stability and fast convergence.
   - Out-of-fold (OOF) predictions are generated for the training set and averaged across folds for the test set.
3. **Acoustic Prosody Extraction (`librosa`):**
   - We extract acoustic features that capture spoken fluency and hesitation:
     - **Words Per Minute (WPM):** Speaking tempo and fluency proxy.
     - **Silence Ratio:** Percentage of the clip spent in silent hesitations (>30dB drop).
     - **RMS Energy (Mean & Std):** Volume dynamics and vocal confidence.
     - **Zero Crossing Rate (ZCR):** Voiced vs. unvoiced acoustic texture.
     - **Duration & Word Count:** Audio duration and vocabulary output volume.
4. **Multimodal Stacking (`LightGBM`):**
   - A 5-fold cross-validated LightGBM regressor fuses DeBERTa's linguistic predictions with the acoustic prosodic descriptors, capturing non-linear interactions between text quality and spoken fluency.

#### 3. Benchmark Progression & Results
- **Dataset Mean Baseline:** RMSE = `1.2390` | Pearson r = `0.0000`
- **DeBERTa-v3 Text Baseline:** Test RMSE = `0.5045` | OOF Pearson r = `0.7910`
- **Multimodal Stacking (DeBERTa + Acoustic + LightGBM):** **Test RMSE = 0.4802** | **OOF Pearson r = 0.8274**
- **Compulsory Training Data RMSE:** `0.4618`


In [ ]:
# Setup and libraries
import os
import re
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BASE_DIR = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR = os.path.join(BASE_DIR, 'test')
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Exploratory Data Analysis & Label Distribution

We inspect the distribution of target grammar scores in the training set (`train.csv`). The labels follow a discrete Likert scale in 0.5 increments ranging from 0.0 to 5.0.


In [ ]:
# Load metadata and check score distribution
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))

print(f"Training samples: {len(train_df)}")
print(f"Testing samples:  {len(test_df)}")

plt.figure(figsize=(9, 4))
sns.countplot(data=train_df, x='label', palette='viridis')
plt.title('Training Data: Grammar Score Distribution (MOS Likert 0.0 - 5.0)')
plt.xlabel('Grammar Score')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"Mean Score: {train_df['label'].mean():.3f} | Std Dev: {train_df['label'].std():.3f}")


## 2. Error-Preserving Speech-to-Text Transcription

Standard ASR models tend to auto-correct grammar mistakes. We configure `faster-whisper-large-v3` with `condition_on_previous_text=False` and `temperature=0.0` to ensure each audio segment is transcribed verbatim, preserving grammatical errors, false starts, and repetitions.


In [ ]:
# Faster-Whisper Transcription Pipeline
from faster_whisper import WhisperModel

asr_model = WhisperModel("large-v3", device="cuda", compute_type="float16")

def transcribe_audio_dataset(df, audio_dir):
    transcripts = []
    for i, row in df.iterrows():
        audio_path = os.path.join(audio_dir, row["filename"])
        segments, _ = asr_model.transcribe(
            audio_path,
            beam_size=5,
            language="en",
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            temperature=0.0
        )
        text = " ".join([seg.text.strip() for seg in segments])
        transcripts.append(text)
    return transcripts

train_df["transcript"] = transcribe_audio_dataset(train_df, TRAIN_DIR)
test_df["transcript"] = transcribe_audio_dataset(test_df, TEST_DIR)

train_df.to_csv(os.path.join(OUT_DIR, "train_transcripts.csv"), index=False)
test_df.to_csv(os.path.join(OUT_DIR, "test_transcripts.csv"), index=False)
print("Transcription completed and cached.")


## 3. Syntactic Text Modeling with DeBERTa-v3

We fine-tune `microsoft/deberta-v3-base` across 5 folds to evaluate the syntactic and grammatical structure of the transcripts:
- **Differential Learning Rates:** We use `1e-4` for the newly initialized linear regression head and `1e-5` for the pretrained encoder layers to prevent catastrophic forgetting while allowing the head to adapt quickly.
- **Full Precision (FP32):** We train in FP32 to ensure numerical stability and prevent gradient overflows common with DeBERTa LayerNorms under FP16.
- **Out-of-Fold (OOF) Generation:** Predictions from all 5 validation folds are stitched together to form a leak-free textual feature for the stacking stage.


In [ ]:
# DeBERTa-v3 5-Fold Training Loop
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 15
LR_ENCODER = 1e-5
LR_HEAD = 1e-4
EARLY_STOPPING = 3
N_FOLDS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GrammarDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits.squeeze(-1)
        loss = F.mse_loss(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)

def predict_model(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop("labels", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits.squeeze(-1)
            preds.extend(logits.cpu().numpy().tolist())
    return np.array(preds)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
texts = train_df["transcript"].tolist()
labels = train_df["label"].tolist()
test_texts = test_df["transcript"].tolist()

oof_deberta = np.zeros(len(train_df))
test_deberta = np.zeros(len(test_df))

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (t_idx, v_idx) in enumerate(kf.split(texts)):
    train_ds = GrammarDataset([texts[i] for i in t_idx], [labels[i] for i in t_idx])
    val_ds = GrammarDataset([texts[i] for i in v_idx], [labels[i] for i in v_idx])
    test_ds = GrammarDataset(test_texts)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    ).to(device).float()

    optimizer = torch.optim.AdamW([
        {"params": model.deberta.parameters(), "lr": LR_ENCODER},
        {"params": model.classifier.parameters(), "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    best_val_rmse = float("inf")
    best_val_preds = None
    best_test_preds = None
    patience = 0

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        val_preds = predict_model(model, val_loader, device)
        val_rmse = np.sqrt(np.mean((np.clip(val_preds, 0.0, 5.0) - np.array([labels[i] for i in v_idx])) ** 2))

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_val_preds = val_preds
            best_test_preds = predict_model(model, test_loader, device)
            patience = 0
            torch.save(model.state_dict(), os.path.join(OUT_DIR, f"deberta_fold{fold+1}.pt"))
        else:
            patience += 1
            if patience >= EARLY_STOPPING:
                break

    oof_deberta[v_idx] = best_val_preds
    test_deberta += best_test_preds / N_FOLDS
    del model; torch.cuda.empty_cache()

train_df["oof_pred"] = np.clip(oof_deberta, 0.0, 5.0)
test_df["oof_pred"] = np.clip(test_deberta, 0.0, 5.0)
train_df.to_csv(os.path.join(OUT_DIR, "train_oof_predictions.csv"), index=False)


## 4. Acoustic & Prosodic Feature Extraction

Human evaluators rate grammar in speech based on both vocabulary structure and spoken fluency. We use `librosa` to extract acoustic features directly from the raw `.wav` audio:
- **Words Per Minute (WPM):** Measures speech articulation speed.
- **Silence Ratio:** Detects intervals below 30dB to quantify hesitation frequency and pause length.
- **RMS Energy (Mean & Std):** Measures vocal dynamics and energy consistency.
- **Zero Crossing Rate (ZCR):** Captures voiced/unvoiced phonetic texture.
- **Duration & Word Count:** Audio length and total words spoken.


In [ ]:
# Extract Acoustic Prosody Features via Librosa
def extract_audio_prosody(audio_path, transcript):
    try:
        y, sr = librosa.load(audio_path, sr=16000)
    except Exception:
        return {"duration": 0.001, "rms_mean": 0.0, "rms_std": 0.0, "zcr_mean": 0.0, "silence_ratio": 1.0, "word_count": 0, "wpm": 0.0}

    duration = max(0.001, librosa.get_duration(y=y, sr=sr))
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y)[0]

    intervals = librosa.effects.split(y, top_db=30)
    non_silent_dur = sum([end - start for start, end in intervals]) / sr
    silence_ratio = np.clip(1.0 - (non_silent_dur / duration), 0.0, 1.0)

    word_count = len(str(transcript).split())
    wpm = word_count / (duration / 60.0)

    return {
        "duration": duration,
        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),
        "zcr_mean": float(np.mean(zcr)),
        "silence_ratio": float(silence_ratio),
        "word_count": word_count,
        "wpm": float(wpm)
    }

train_audio_feats = [extract_audio_prosody(os.path.join(TRAIN_DIR, r["filename"]), r["transcript"]) for _, r in train_df.iterrows()]
test_audio_feats = [extract_audio_prosody(os.path.join(TEST_DIR, r["filename"]), r["transcript"]) for _, r in test_df.iterrows()]

train_feat_df = pd.concat([train_df, pd.DataFrame(train_audio_feats)], axis=1)
test_feat_df = pd.concat([test_df, pd.DataFrame(test_audio_feats)], axis=1)

train_feat_df.to_csv(os.path.join(OUT_DIR, "train_audio_features.csv"), index=False)
test_feat_df.to_csv(os.path.join(OUT_DIR, "test_audio_features.csv"), index=False)
print("Acoustic feature extraction completed.")


## 5. Multimodal Stacking Regressor (LightGBM)

We train a LightGBM Gradient Boosted Regressor on the combined feature space:
- **Input Features:** DeBERTa Out-of-Fold linguistic prediction (`oof_pred`) + 7 acoustic prosodic descriptors (`duration`, `rms_mean`, `rms_std`, `zcr_mean`, `silence_ratio`, `word_count`, `wpm`).
- **5-Fold Cross Validation:** Matches the fold splits from earlier stages to prevent data leakage.
- **Rule-based Silence Override:** Automatically sets predictions to `0.0` for clips with 0 words or silence ratio > 95%.


In [ ]:
# LightGBM Multimodal Stacking Model
FEATURES = ["oof_pred", "duration", "rms_mean", "rms_std", "zcr_mean", "silence_ratio", "word_count", "wpm"]
X = train_feat_df[FEATURES]
y = train_feat_df["label"]
X_test = test_feat_df[FEATURES]

oof_lgb = np.zeros(len(train_feat_df))
test_lgb = np.zeros(len(test_feat_df))

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 15,
    'max_depth': 4,
    'feature_fraction': 0.8,
    'random_state': SEED,
    'n_estimators': 500,
    'verbose': -1
}

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_train, y_train = X.iloc[t_idx], y.iloc[t_idx]
    X_val, y_val = X.iloc[v_idx], y.iloc[v_idx]

    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_lgb[v_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(X_test) / N_FOLDS

def apply_silence_override(df, preds):
    silence_mask = (df["word_count"] == 0) | (df["rms_mean"] < 0.001) | (df["silence_ratio"] > 0.95)
    return np.where(silence_mask, 0.0, preds)

oof_lgb = np.clip(apply_silence_override(train_feat_df, oof_lgb), 0.0, 5.0)
test_lgb = np.clip(apply_silence_override(test_feat_df, test_lgb), 0.0, 5.0)


## 6. Evaluation Results Training RMSE & Visualizations


Below we compute the training RMSE and plot three diagnostic visualizations:
1. **Actual vs. Predicted Regression Plot:** Visualizes alignment with ground truth and computes Pearson correlation ($r$).
2. **Feature Importance Bar Chart:** Shows the relative contribution of DeBERTa text predictions vs. acoustic features.
3. **Residual Distribution:** Validates that model errors are centered around zero with low variance.


In [ ]:
# Evaluation Metrics and Diagnostics
train_rmse = np.sqrt(np.mean((oof_lgb - y) ** 2))
train_pearson = pearsonr(y, oof_lgb)[0]

print("=" * 60)
print(f"  COMPULSORY TRAINING DATA RMSE: {train_rmse:.4f}")
print(f"  CROSS-VALIDATION PEARSON (r):   {train_pearson:.4f}")
print("=" * 60)

# 1. Actual vs Predicted Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.regplot(x=y, y=oof_lgb, scatter_kws={'alpha':0.4, 'color':'royalblue'}, line_kws={'color':'crimson'})
plt.title(f'Actual vs Predicted Scores (Pearson r = {train_pearson:.4f})')
plt.xlabel('Ground Truth Grammar Score')
plt.ylabel('Predicted Grammar Score')
plt.grid(True, alpha=0.3)

# 2. Feature Importance Chart
plt.subplot(1, 2, 2)
feat_imp = pd.DataFrame({'Feature': FEATURES, 'Importance': lgb_model.feature_importances_}).sort_values('Importance', ascending=True)
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='teal')
plt.title('LightGBM Multimodal Feature Importance')
plt.xlabel('Importance')
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Residual Distribution
plt.figure(figsize=(7, 4))
residuals = oof_lgb - y
sns.histplot(residuals, kde=True, color='darkorange', bins=30)
plt.title(f'Prediction Residual Distribution (Mean = {residuals.mean():.3f}, Std = {residuals.std():.3f})')
plt.xlabel('Residual (Predicted - Actual)')
plt.ylabel('Density')
plt.grid(True, alpha=0.3)
plt.show()


## 7. Submission File Generation

We generate the final submission file (`submission_lgb.csv`) conforming to the competition format.


In [ ]:
# Generate final submission file
submission = test_feat_df[["filename"]].copy()
submission["label"] = test_lgb

submission_path = os.path.join(OUT_DIR, "submission_lgb.csv")
submission.to_csv(submission_path, index=False)

print(f"Final submission saved to: {submission_path}")
print("
Sample predictions:")
display(submission.head(10))
